In [1]:
# 1. Install OpenAI CLIP
!pip install git+https://github.com/openai/CLIP.git

# 2. Create the checkpoints folder
!mkdir -p ./checkpoints

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-awqyuzdc
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-awqyuzdc
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.9 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=8aa5cfdaa2331963eda3b7d18be48f935cd39f42b44f268939a6d47da459cafe
  Stored in directory: /tmp/pip-ephem-wheel-cache-ldtj00u9/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [2]:
import os
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from tqdm import tqdm

# ==========================================
# 1. NaN-Safe DistArc Loss Module
# ==========================================
class DistArcLoss(nn.Module):
    def __init__(self, in_features, out_features, m=0.4, lam=0.005, radial_gap=0.5):
        super(DistArcLoss, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.m = m
        self.lam = lam
        
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        
        self.W = nn.Parameter(torch.FloatTensor(in_features, out_features))
        nn.init.kaiming_uniform_(self.W)

        radii = np.arange(2.0, 2.0 + out_features * radial_gap, radial_gap).astype('float32')
        self.register_buffer('r', torch.tensor(radii))

    def forward(self, x, labels):
        batch_size = x.size(0)
        
        x_norm = F.normalize(x, p=2, dim=1)
        W_norm = F.normalize(self.W, p=2, dim=0)
        
        cosine = torch.matmul(x_norm, W_norm).clamp(-1.0 + 1e-7, 1.0 - 1e-7)
        target_logit = cosine[torch.arange(batch_size), labels]
        
        sine = torch.sqrt((1.0 - torch.pow(target_logit, 2)).clamp(min=1e-7))
        marginal_cosine = target_logit * self.cos_m - sine * self.sin_m 

        W_scaled = W_norm * self.r
        target_w_r = W_scaled[:, labels].T 
        R = -target_w_r + x 
        
        R_norm = F.normalize(R, p=2, dim=1)
        target_w_r_norm = F.normalize(-target_w_r, p=2, dim=1)
        cos_phi = torch.sum(R_norm * target_w_r_norm, dim=1) 

        x_sq = torch.sum(x**2, dim=1, keepdim=True) 
        w_r_sq = torch.sum(W_scaled**2, dim=0, keepdim=True) 
        dist_sq = x_sq + w_r_sq - 2 * torch.matmul(x, W_scaled) 

        num_exp = marginal_cosine + cos_phi - (self.lam * dist_sq[torch.arange(batch_size), labels])
        
        denom_terms = cosine - (self.lam * dist_sq)
        denom_terms[torch.arange(batch_size), labels] = marginal_cosine
        
        loss = -num_exp + torch.logsumexp(denom_terms, dim=1)
        return loss.mean()

    def predict(self, x):
        x = x.float()
        W_scaled = F.normalize(self.W, p=2, dim=0) * self.r
        x_sq = torch.sum(x**2, dim=1, keepdim=True)
        w_r_sq = torch.sum(W_scaled**2, dim=0, keepdim=True)
        dist_sq = x_sq + w_r_sq - 2 * torch.matmul(x, W_scaled)
        return torch.argmin(dist_sq, dim=1)

# ==========================================
# 2. ACTUAL iResNet50 Architecture
# ==========================================
def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=dilation, groups=groups, bias=False, dilation=dilation)

def conv1x1(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)

class IBasicBlock(nn.Module):
    expansion = 1
    def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1, base_width=64, dilation=1):
        super(IBasicBlock, self).__init__()
        self.bn1 = nn.BatchNorm2d(inplanes, eps=1e-05)
        self.conv1 = conv3x3(inplanes, planes)
        self.bn2 = nn.BatchNorm2d(planes, eps=1e-05)
        self.prelu = nn.PReLU(planes)
        self.conv2 = conv3x3(planes, planes, stride)
        self.bn3 = nn.BatchNorm2d(planes, eps=1e-05)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x
        out = self.bn1(x)
        out = self.conv1(out)
        out = self.bn2(out)
        out = self.prelu(out)
        out = self.conv2(out)
        out = self.bn3(out)
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        return out

class IResNet(nn.Module):
    fc_scale = 7 * 7 # Expects 112x112 input
    def __init__(self, block, layers, dropout=0, num_features=512, zero_init_residual=False, groups=1, width_per_group=64):
        super(IResNet, self).__init__()
        self.inplanes = 64
        self.dilation = 1
        self.groups = groups
        self.base_width = width_per_group
        
        self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(self.inplanes, eps=1e-05)
        self.prelu = nn.PReLU(self.inplanes)
        self.layer1 = self._make_layer(block, 64, layers[0], stride=2)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.bn2 = nn.BatchNorm2d(512 * block.expansion, eps=1e-05)
        self.dropout = nn.Dropout(p=dropout, inplace=True)
        self.fc = nn.Linear(512 * block.expansion * self.fc_scale, num_features)
        self.features = nn.BatchNorm1d(num_features, eps=1e-05)
        nn.init.constant_(self.features.weight, 1.0)
        self.features.weight.requires_grad = False

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                nn.BatchNorm2d(planes * block.expansion, eps=1e-05),
            )
        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample, self.groups, self.base_width, self.dilation))
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes, groups=self.groups, base_width=self.base_width, dilation=self.dilation))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.prelu(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.bn2(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        x = self.features(x)
        return x

def iresnet50(**kwargs):
    return IResNet(IBasicBlock, [3, 4, 14, 3], **kwargs)

# ==========================================
# 3. Model Wrapper
# ==========================================
class iResNet50_HyperSpaceX(nn.Module):
    def __init__(self, embedding_size=2, num_classes=100, loss_type='dist_arc'):
        super(iResNet50_HyperSpaceX, self).__init__()
        
        self.backbone = iresnet50(num_features=512)
        self.projection = nn.Linear(512, embedding_size)
        self.loss_type = loss_type
        
        if self.loss_type == 'cross_entropy':
            self.classifier = nn.Linear(embedding_size, num_classes)
            
    def forward(self, x):
        x = self.backbone(x) 
        embeddings = self.projection(x)
        
        if self.loss_type == 'cross_entropy':
            return self.classifier(embeddings)
        return embeddings

# ==========================================
# 4. Data Loading 
# ==========================================
def get_dataloaders(batch_size):
    preprocess = transforms.Compose([
        transforms.Resize((112, 112)), 
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

    train_dataset = torchvision.datasets.CIFAR100(root='./data', train=True, transform=preprocess, download=True)
    val_dataset = torchvision.datasets.CIFAR100(root='./data', train=False, transform=preprocess, download=True)
    
    # FIX: persistent_workers=True and reduced num_workers to prevent shared memory leaks over many epochs
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, 
                                               num_workers=2, pin_memory=True, persistent_workers=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False, 
                                             num_workers=2, pin_memory=True, persistent_workers=True)
    return train_loader, val_loader

# ==========================================
# 5. Main Training Loop
# ==========================================
def train_model(
    embedding_size=512, 
    loss_type='dist_arc', 
    epochs=150, 
    batch_size=128, 
    lr=1e-2,           
    weight_decay=5e-4, 
    lam=0.005,
    save_dir='./checkpoints',
    resume_path=None # FIX: Added explicit path to resume from downloaded models
):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    num_classes = 100 
    
    os.makedirs(save_dir, exist_ok=True)
    
    best_save_path = os.path.join(save_dir, f'best_iresnet50_dim{embedding_size}_{loss_type}.pth')
    latest_save_path = os.path.join(save_dir, f'latest_iresnet50_dim{embedding_size}_{loss_type}.pth')

    print(f"--- Starting AMP Training ---")
    print(f"Model: True iResNet50 | Emb Size: {embedding_size} | Loss: {loss_type} | Target: CIFAR-100")

    model = iResNet50_HyperSpaceX(embedding_size, num_classes, loss_type).to(device)
    train_loader, val_loader = get_dataloaders(batch_size)

    if torch.cuda.device_count() > 1:
        print(f"Awesome, utilizing {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)
    
    if loss_type == 'dist_arc':
        criterion = DistArcLoss(embedding_size, num_classes, lam=lam).to(device)
        optimizer = optim.SGD([
            {'params': model.parameters()},
            {'params': criterion.parameters()}
        ], lr=lr, weight_decay=weight_decay, momentum=0.9)
    else:
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay, momentum=0.9)

    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=75, gamma=0.1)
    scaler = torch.amp.GradScaler('cuda')
    
    best_acc = 0.0
    start_epoch = 0

    # FIX: Robust resuming logic from specific file
    target_load_path = resume_path if resume_path else latest_save_path
    
    if target_load_path and os.path.exists(target_load_path):
        print(f"Found existing checkpoint! Loading from {target_load_path}...")
        checkpoint = torch.load(target_load_path, map_location=device)
        
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        
        if 'scheduler_state_dict' in checkpoint:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            
        if loss_type == 'dist_arc' and 'criterion_state_dict' in checkpoint:
            criterion.load_state_dict(checkpoint['criterion_state_dict'])
            
        start_epoch = checkpoint['epoch']
        best_acc = checkpoint.get('best_acc', 0.0)
        print(f"Resuming training from Epoch {start_epoch + 1} with Best Acc so far: {best_acc:.2f}%")

    for epoch in range(start_epoch, epochs):
        
        model.train()
        if loss_type == 'dist_arc': criterion.train()
        
        running_loss = 0.0
        
        # FIX: Added leave=False and mininterval to stop log-file bloat. 
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False, mininterval=2.0)
        
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            
            with torch.amp.autocast('cuda'):
                outputs = model(images)
            
            if loss_type == 'dist_arc':
                loss = criterion(outputs.float(), labels)
            else:
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            
            # FIX: Included criterion.parameters() in the clipping step
            params_to_clip = list(model.parameters())
            if loss_type == 'dist_arc':
                params_to_clip += list(criterion.parameters())
                
            torch.nn.utils.clip_grad_norm_(params_to_clip, max_norm=5.0)
            
            scaler.step(optimizer)
            scaler.update()
            
            running_loss += loss.item()
            pbar.set_postfix({'loss': f"{running_loss/(pbar.n+1):.4f}"})
            
        model.eval()
        if loss_type == 'dist_arc': criterion.eval()
        
        correct = 0
        total = 0
        with torch.no_grad():
            # FIX: Throttled val tqdm
            val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False, mininterval=2.0)
            for images, labels in val_pbar:
                images, labels = images.to(device), labels.to(device)
                
                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                
                outputs = outputs.float()
                
                if loss_type == 'cross_entropy':
                    _, predicted = torch.max(outputs, 1)
                else:
                    predicted = criterion.predict(outputs)
                    
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
        acc = 100 * correct / total
        
        scheduler.step()
        
        # Clean print block that won't bloat the notebook JSON
        print(f"Epoch {epoch+1}/{epochs} Complete | Train Loss: {running_loss/len(train_loader):.4f} | Val Acc: {acc:.2f}% | Next LR: {scheduler.get_last_lr()[0]}")
        
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_acc': best_acc,
            'loss_type': loss_type,
            'embedding_size': embedding_size
        }
        if loss_type == 'dist_arc':
            checkpoint['criterion_state_dict'] = criterion.state_dict()
            
        torch.save(checkpoint, latest_save_path)
        
        if acc > best_acc:
            best_acc = acc
            print(f"--> New Best Accuracy! ({best_acc:.2f}%) Saving BEST model...")
            torch.save(checkpoint, best_save_path)

if __name__ == '__main__':
    # ---------------------------------------------------------
    # Upload your downloaded file to the Kaggle/Colab directory 
    # and update the path below to point to it.
    # ---------------------------------------------------------
    my_downloaded_model_path = '/kaggle/input/models/varareddy2527/best-iresnet50-dim512-cross-entropy/pytorch/default/1/best_iresnet50_dim512_cross_entropy.pth' 
    
    train_model(
        embedding_size=512, 
        loss_type='cross_entropy', 
        epochs=150, 
        batch_size=128, 
        lr=1e-2, 
        resume_path=my_downloaded_model_path
    )

    train_model(
        embedding_size=512, 
        loss_type='dist_arc', 
        epochs=150, 
        batch_size=128, 
        lr=1e-2,
    )

--- Starting AMP Training ---
Model: True iResNet50 | Emb Size: 512 | Loss: cross_entropy | Target: CIFAR-100


100%|██████████| 169M/169M [00:10<00:00, 15.6MB/s]


Awesome, utilizing 2 GPUs!
Found existing checkpoint! Loading from /kaggle/input/models/varareddy2527/best-iresnet50-dim512-cross-entropy/pytorch/default/1/best_iresnet50_dim512_cross_entropy.pth...
Resuming training from Epoch 51 with Best Acc so far: 61.28%


Epoch 51/150 Complete | Train Loss: 0.0054 | Val Acc: 61.46% | Next LR: 0.01
--> New Best Accuracy! (61.46%) Saving BEST model...


Epoch 52/150 Complete | Train Loss: 0.0055 | Val Acc: 61.77% | Next LR: 0.01
--> New Best Accuracy! (61.77%) Saving BEST model...


...

Epoch 104/150 Complete | Train Loss: 0.0022 | Val Acc: 61.06% | Next LR: 0.001


Epoch 105/150 Complete | Train Loss: 0.0022 | Val Acc: 60.96% | Next LR: 0.001


Epoch 106/150 Complete | Train Loss: 0.0022 | Val Acc: 61.10% | Next LR: 0.001


Epoch 107/150 Complete | Train Loss: 0.0022 | Val Acc: 61.04% | Next LR: 0.001


Epoch 108/150 [Train]:  78%|███████▊  | 305/391 [05:00<08:48,  6.14s/it, loss=0.0021]IOStream.flush timed out
